# 06 -- TS-ICL zero-shot imputation: the real API

**Purpose**: show the actual TS-ICL calling convention used elsewhere in this
repository (`demo/src/methods.py`, `src/coastal_gap_reconstruction/tsicl_helpers.py`),
for target-only and covariate-conditioned inputs.
**Inputs**: the public chlorophyll daily target and predictor feature tables.
**Outputs**: a point estimate and quantile band for one illustrative masked window
(only if TS-ICL is installed in the current environment -- see below).
**Execution status**: the setup/data cells always run. The live TS-ICL cells run
only if `tsicl` is installed and its checkpoint is reachable; otherwise they
print a clear skip message rather than failing the notebook.

For the full, always-executable, visual, multi-configuration comparison
(target-only vs. +satellite chlorophyll vs. +wind/SST, on a real 14-day gap,
plotted against withheld ground truth), see
`demo/gap_reconstruction_walkthrough.ipynb` and run `bash demo/run_demo.sh`.
This notebook is a short API reference, not a second copy of that walkthrough.


## Installing TS-ICL

TS-ICL (https://github.com/EDF-Lab/ts-icl) is installed from PyPI:

```bash
pip install tsicl torch
```

On first use it downloads its public, non-gated pretrained checkpoint
(~209 MB) from a public Hugging Face repository and caches it locally. It is
released under the TS-ICL Non-Commercial License v1.0 (EDF SA), which permits
non-commercial research, evaluation, and benchmarking on third-party data --
review the license yourself before use. This repository does not vendor
TS-ICL; `src/coastal_gap_reconstruction/tsicl_helpers.py` is only a thin
calling-convention helper, and `demo/run_demo.sh` builds an isolated
environment with the real dependency for you.


In [ ]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
sys.path.insert(0, "../demo")

from coastal_gap_reconstruction.data_loading import load_daily_target, load_feature_table
from src import methods as mth  # demo's tested TS-ICL calling layer

target_df = load_daily_target("../data_public/chlorophyll/chlorophyll_daily_target.csv")
features_df = load_feature_table("../data_public/chlorophyll/chlorophyll_predictor_features_curated.csv")

# TS-ICL operates in log10 space, as used throughout this benchmark.
log10_chl = np.log10(target_df["chl_mean"].clip(lower=1e-3))


## Loading the model

`load_tsicl()` never raises: it returns `(None, status)` with `status.live =
False` if `tsicl`/`torch` are not installed or the checkpoint cannot be
fetched, so this notebook degrades gracefully instead of failing outright.


In [ ]:
model, status = mth.load_tsicl()
print(status)

if not status.live:
    print(
        "TS-ICL is not available in this environment "
        "(pip install tsicl torch, or run demo/run_demo.sh for an isolated "
        "environment that installs it). The remaining cells are skipped."
    )


## Target-only imputation

Tensor shapes: `inputs` is a 1D float32 tensor of length T with `NaN` marking
the positions to reconstruct; `covars`, when supplied, is `(1, T, C)` --
batch dimension first, one series. `model.impute(...)` returns a mean
prediction and a set of quantile predictions, both length T, already
denormalized back to log10-chlorophyll space.


In [ ]:
if status.live:
    example_start, example_end = "2018-03-01", "2018-03-14"
    window = log10_chl.loc["2018-02-20":"2018-03-20"].copy()
    masked = window.copy()
    masked.loc[example_start:example_end] = np.nan

    mean, quantiles = mth._tsicl_impute(model, masked.to_numpy(), covar_array=None)

    result = pd.DataFrame(
        {
            "date": window.index,
            "pred_chl": 10 ** mean,
            "q05": 10 ** quantiles[:, 0],
            "q95": 10 ** quantiles[:, -1],
        }
    )
    result.loc[result["date"].between(example_start, example_end)]


## Covariate-conditioned imputation

Same call, with a satellite chlorophyll proxy passed as a single covariate
channel. `build_covariate_block` in `tsicl_helpers.py` documents the one
non-obvious shape requirement (an explicit batch dimension of size 1).


In [ ]:
if status.live:
    from coastal_gap_reconstruction.tsicl_helpers import build_covariate_block

    satellite_proxy = features_df["chl_cons_log10"].loc["2018-02-20":"2018-03-20"]
    covar_block = build_covariate_block(satellite_proxy.to_numpy()[:, None])[0]  # (T, C) for _tsicl_impute

    mean_cov, quantiles_cov = mth._tsicl_impute(model, masked.to_numpy(), covar_array=covar_block)

    result_cov = pd.DataFrame(
        {
            "date": window.index,
            "pred_chl": 10 ** mean_cov,
            "q05": 10 ** quantiles_cov[:, 0],
            "q95": 10 ** quantiles_cov[:, -1],
        }
    )
    result_cov.loc[result_cov["date"].between(example_start, example_end)]


## Notes

- Evaluate TS-ICL outputs only at positions that were actually masked in a
  proper artificial-gap validation run (see
  `notebooks/02_artificial_gap_validation.ipynb`) -- the single window above
  is illustrative of the API only, not a validation result.
- Never treat a prediction on a real (naturally occurring) gap as validation
  evidence -- see `docs/evidence_hierarchy.md`.
- For the full benchmark numbers this repository reports for TS-ICL, see
  `results_public/chlorophyll/chlorophyll_benchmark_summary.csv`,
  `results_public/oxygen/oxygen_benchmark_by_length.csv`, and
  `docs/methodology/tsicl_usage.md`.
